# ДЗ №2. Отбор кандидатов


Общая информация
Дата выдачи: 3 февраля 2026

Дедлайн: 23 февраля 2026 23:59 MSK


В этом и последующих домашних заданиях мы построим приближенную к реальной рекомендательную систему. Работать будем с данными marketplace из [T-ECD](https://huggingface.co/datasets/t-tech/T-ECD).

Обычно рекомендательная система состоит из нескольких этапов:
1. Отбор кандидатов (Retrieval)
2. Ранжирование (Ranking)
3. Бизнес-логика (например, условие на то, чтобы товары от одного продавца не стояли в ленте друг за другом)

В этом домашнем задании сосредоточимся на первом этапе.

По ресурсам: at worst нужно 15GB RAM и 3.5GB свободного дискового пространства, GPU не требуется (но можно, если хочется). Дз спокойно запускается на ресурсах Kaggle-а.

In [4]:
!pip install -q polars huggingface_hub implicit faiss-cpu torch


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3.10 -m pip install --upgrade pip


In [1]:
import glob
import os
import random
import time
import gc
from abc import ABC, abstractmethod
from collections import defaultdict, deque
from math import log2

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from implicit.als import AlternatingLeastSquares
from implicit.ann.faiss import FaissModel
from IPython.display import HTML
from scipy.sparse import coo_matrix, csr_matrix
from torch.utils.data import DataLoader, Dataset, IterableDataset
from tqdm.notebook import tqdm

## Download Data

Данные занимают около 3.5 GB

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

## EDA (1p)

Изучите данные (users, items, events). Для работы используйте polars - это более оптимизированный аналог pandas. Обратите внимание, что в polars существуют eager и lazy методы работы с таблицами, и зачастую эффективнее использовать lazy.

1. Оцените количество строк и столбцов в таблицах
2. Для всех категориальных переменных рассчитайте частотность каждого признака и процент null-ов. Удобно вывести результат в виде таблицы с помощью следующего кода: `HTML(polars_df.to_pandas().to_html())` - так не нужно будет задумываться про `display.max_rows`
3. Постройте гистограмму для числовых переменных, также оцените процент null-ов
4. Постройте график, по оси X которого будет отмечен номер файла events (в порядке возрастания), а по оси Y количество уникальных пользователей и айтемов за этот день
5. Постройте график, по оси X которого будет отмечен номер файла events (в порядке возрастания), а по оси Y количество событий в разбивке по `action_type`
6. Кликаутят (переходят на сайт партнера) ли люди один и тот же товар несколько раз?
7. Опционально любая другая аналитика на ваше усмотрение (например, можно покластеризовать эмбеддинги товаров и сравнить пересечения кластеров с категориями)

In [2]:
# uppercase для global переменных, чтобы легче видеть их в коде
USERS = pl.read_parquet("dataset/small/users.pq")
ITEMS = pl.read_parquet("dataset/small/marketplace/items.pq")
CATALOG_SIZE = len(ITEMS)

### Users

In [ ]:
# YOUR CODE HERE

### Items

In [ ]:
# YOUR CODE HERE

### Events

In [ ]:
# YOUR CODE HERE

## Data Splitting (1p)

Прежде чем обучать какие-либо модели, разобъем данные на train-val-test по временной отметке (темпоральный сплит).

На практике обычно используется следующий подход: сначала модель обучается на train части, на val подбирается лучший набор гиперпараметров. Потом модель обучается на train+val, и финальные метрики замеряются на test.

Допишите функцию так, чтобы:

* в train вошли данные start_day < day <= start_day + n_train_days
* в val вошли данные start_day + n_train_days < day <= start_day + n_train_days + n_val_days
* в test вошли данные start_day + n_train_days + n_val_days < day <= start_day + n_train_days + n_val_days + n_test_days

Где start_day - это номер файла, задающий дату, с которой начинается временной промежуток.

Не будем учитывать повторные взаимодействия в обучающих данных. То есть для каждой пары (пользователь, айтем) в train/train_val необходимо оставить только максимальный лейбл.

Ожидаемую результирующую схему данных можно найти ниже. В val и test нужно оставить только позитивы.

In [3]:
def get_data(
    start_day: int = 1250,
    n_train_days: int = 20,
    n_val_days: int = 3,
    n_test_days: int = 5,
    n_users_to_keep: int = 10_000,
    n_items_to_keep: int = 30_000,
) -> tuple[pl.LazyFrame, pl.LazyFrame, pl.LazyFrame, pl.LazyFrame]:

    events = (
        pl.scan_parquet("dataset/small/marketplace/events/")
        .with_columns(
            label=(
                pl.when(pl.col("action_type") == "view")
                .then(0)
                .otherwise(1)
            )  # `label` равен 1, если `action_type` НЕ равен просмотру, и 0 иначе
        )
        .filter(
            (pl.col("timestamp").dt.total_days() > start_day) &
            (pl.col("timestamp").dt.total_days() <= start_day + n_train_days + n_val_days + n_test_days)
        )
        .sort("timestamp")
        .select("timestamp", "user_id", "item_id", "label")
    )

    # Отфильтруем топ-`n_users_to_keep` пользователей и топ-`n_items_to_keep` айтемов по количеству положительных взаимодействий
    # Это нужно, чтобы не обучать модели на слишком больших матрицах (потому что у нас тут всё-таки дз, а не полноценный продовый сетап)
    
    user_positive_counts = (
        events
        .group_by("user_id")
        .agg(pl.col("label").sum())
        .sort("label", descending=True)
        .head(n_users_to_keep)
        .select("user_id")
    )
    
    item_counts = (
        events
        .group_by("item_id")
        .agg(pl.col("label").sum())
        .sort("label", descending=True)
        .head(n_items_to_keep)
        .select("item_id")
    )

    # Кстати, у такого способа фильтрации точно есть проблемы - как минимум то, что что статистика для айтемов считается и по тем юзерам, 
    # которые не попали в топ-`n_users_to_keep`. Но в качестве приближения работает вполне неплохо 
    
    events_filtered = (
        events
        .join(user_positive_counts, on="user_id", how="inner")
        .join(item_counts, on="item_id", how="inner")
        .collect()
        .lazy()
    )

    train_events = # YOUR CODE HERE

    val_events = # YOUR CODE HERE
    
    train_val_events = # YOUR CODE HERE
    
    test_events = # YOUR CODE HERE

    return (
        train_events, 
        val_events, 
        train_val_events, 
        test_events
    )


train_events, val_events, train_val_events, test_events = get_data()

train_events = train_events.collect()
val_events = val_events.collect()
train_val_events = train_val_events.collect()
test_events = test_events.collect()


In [4]:
expected_train_schema = pl.Schema({
    "user_id": pl.UInt64,
    "item_id": pl.String,
    "timestamp": pl.Duration(time_unit='us'),
    "label": pl.Int32
})

expected_val_schema = pl.Schema({
    "user_id": pl.UInt64,
    "item_id": pl.List(pl.String)
})

assert train_events.schema == expected_train_schema
assert val_events.schema == expected_val_schema
assert train_val_events.schema == expected_train_schema
assert test_events.schema == expected_val_schema

Отфильтруйте из валидационного и тестового подмножеств только теплых пользователей (то есть таких, у которых было хотя бы одно взаимодействие в train или train_val части соответственно). Почему так? Модели, которые рассматриваются в данном дз, не умеют работать с холодными пользователями. 

Кстати, сколько получается таких холодных пользователей?

In [ ]:
# YOUR CODE HERE

In [7]:
assert len(train_events) == 693176
assert len(val_events) == 2375
assert len(train_val_events) == 782473
assert len(test_events) == 3488

In [10]:
assert round(train_events["label"].mean(), 2) == 0.12
assert round(train_val_events["label"].mean(), 2) == 0.12

In [13]:
val_users = val_events["user_id"].to_list()
test_users = test_events["user_id"].to_list()

## Evaluation (1p)

In [14]:
# сколько рекомендаций будем выдавать для каждого пользователя. можете выбрать другое число 
N = 20

### Metrics (0.5p)

Реализуйте набор метрик для оценки качества моделей. Обязательно реализовать HitRate@k (можно взять из прошлой домашки) и Coverage@k. NDCG@k и Recall@k уже реализованы. По желанию можете добавить и другие метрики, например MAP@k, MRR@k и Precision@k.

Условия на векторизацию/обязательность использования polars-а в метриках нет :)

In [15]:
def hit_rate(recommendations, ground_truth, k=100):
    """
    Args:
        recommendations: dict {user_id: list of recommended item_ids}
        ground_truth: dict {user_id: set of relevant item_ids}
        k: cutoff level
    """
    # YOUR CODE HERE


def recall(recommendations, ground_truth, k=100):
    """
    Args:
        recommendations: dict {user_id: list of recommended item_ids}
        ground_truth: dict {user_id: set of relevant item_ids}
        k: cutoff level
    """
    recalls = []

    for user_id, recs in recommendations.items():
        user_recs = recs[:k]
        user_truth = ground_truth.get(user_id, set())

        if not user_truth:  # If no ground truth items, recall is 0
            recalls.append(0.0)
            continue

        relevant_count = sum(1 for item in user_recs if item in user_truth)
        user_recall = relevant_count / len(user_truth)
        recalls.append(user_recall)

    return np.mean(recalls) if recalls else 0.0


def ndcg(recommendations, ground_truth, k=100):
    """
    Args:
        recommendations: dict {user_id: list of recommended item_ids}
        ground_truth: dict {user_id: set of relevant item_ids}
        k: cutoff level for evaluation
    """
    def dcg(scores):
        return np.sum(
            np.divide(
                np.power(2, scores) - 1,
                np.log2(np.arange(scores.shape[0], dtype=np.float64) + 2)
            ),
            dtype=np.float64
        )
    
    ndcg_scores = []
    
    for user_id, pred_items in recommendations.items():
        pred = pred_items[:k]
        gt_set = ground_truth.get(user_id, set())
        
        # Создаем бинарный вектор релевантности для рекомендованных элементов
        # Размер: min(k, len(pred)) - мы оцениваем только первые k рекомендованных
        at = len(pred)
        
        # Для DCG: релевантность рекомендованных элементов
        relevance = np.array([1.0 if item in gt_set else 0.0 for item in pred], dtype=np.float64)
        
        rank_dcg = dcg(relevance)
        
        if rank_dcg == 0.0:
            ndcg_scores.append(0.0)
            continue
        
        # Для IDCG: идеальный случай - все релевантные элементы в начале
        # Берем все релевантные элементы (из ground truth)
        # Но не больше, чем мы можем разместить в рекомендациях длины at
        # Создаем массив из 1 для релевантных и 0 для нерелевантных, сортируем по убыванию
        all_relevances = np.ones(len(gt_set), dtype=np.float64)
        ideal_relevance = np.sort(np.concatenate([all_relevances, np.zeros(max(0, at - len(gt_set)))]))[::-1][:at]
        
        # Вычисляем IDCG
        ideal_dcg = dcg(ideal_relevance)
        
        if ideal_dcg == 0.0:
            ndcg_scores.append(0.0)
        else:
            ndcg_ = rank_dcg / ideal_dcg
            ndcg_scores.append(ndcg_)

    return np.mean(ndcg_scores) if ndcg_scores else 0.0
    

def coverage(recommendations, catalog_size=CATALOG_SIZE, k=100):
    """
    Args:
       recommendations: dict {user_id: list of recommended item_ids}
       catalog_size: total number of unique items in the catalog
       k: cutoff level
    """
    # YOUR CODE HERE


Микро-тест:

In [16]:
recommendations = {
    'user1': ['item1', 'item2', 'item3', 'item4', 'item5'],
    'user2': ['item2', 'item1', 'item4', 'item5', 'item3'],
    'user3': ['item5', 'item4', 'item3', 'item2', 'item1'],
    'user4': ['item6', 'item7', 'item8', 'item9', 'item10'],
}
ground_truth = {
    'user1': {'item1', 'item3'},
    'user2': {'item1', 'item2', 'item4'},
    'user3': {'item5'},
    'user4': set(),
}

hr_val = hit_rate(recommendations, ground_truth, 3)
assert np.isclose(hr_val, 0.75, atol=1e-2), f"Expected HitRate 0.75, got {hr_val}"

rec_val = recall(recommendations, ground_truth, 3)
assert np.isclose(rec_val, 0.75, atol=1e-2), f"Expected Recall 0.75, got {rec_val}"

ndcg_val = ndcg(recommendations, ground_truth, 3)
assert np.isclose(ndcg_val, 0.73, atol=1e-2), f"Expected NDCG ~0.73, got {ndcg_val}"

coverage_val = coverage(recommendations, catalog_size=20, k=10)
assert np.isclose(coverage_val, 0.5, atol=1e-2), f"Expected Coverage ~0.5, got {coverage_val}"

Создадим функцию, которая будет приводить polars-таблицу с рекомендациями в удобный для расчета метрик вид. Если вы добавляли свои метрики - добавьте их и сюда тоже. 

In [17]:
def _evaluate(recommendations, ground_truth, k):
    results = {}
    
    for k_ in k:
        hr = hit_rate(recommendations, ground_truth, k=k_)
        rec = recall(recommendations, ground_truth, k=k_)
        ndcg_score = ndcg(recommendations, ground_truth, k=k_)
        coverage_score = coverage(recommendations, k=k_)
        
        results[f"hit_rate@{k_}"] = hr
        results[f"recall@{k_}"] = rec
        results[f"ndcg@{k_}"] = ndcg_score
        results[f"coverage@{k_}"] = coverage_score
    
    return results


def evaluate(
    events: pl.DataFrame, 
    recommendations_column: str, 
    ground_truth_column: str = "item_id", 
    k=[5, 10, 20]
):
    recommendations = dict(events.select("user_id", recommendations_column).iter_rows())
    ground_truth = dict(events.select("user_id", ground_truth_column).iter_rows())
    return _evaluate(recommendations, ground_truth, k=k)

Общая табличка с метриками. Будем собирать её шаг за шагом:

In [18]:
%%time

test_events = test_events.with_columns(aa=pl.col("item_id").list.slice(0, N))
metrics_aa = evaluate(test_events, recommendations_column="aa")
RESULTS = pd.concat([
    pd.DataFrame(metrics_aa, index=["A/A"]),  # sanity check
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

CPU times: user 453 ms, sys: 15.7 ms, total: 469 ms
Wall time: 517 ms


,hit_rate@5,recall@5,ndcg@5,coverage@5,hit_rate@10,recall@10,ndcg@10,coverage@10,hit_rate@20,recall@20,ndcg@20,coverage@20
A/A,1.00000,0.82196,1.00000,0.00209,1.00000,0.96464,1.00000,0.00274,1.00000,0.99753,1.00000,0.00299


Как думаете, зачем может быть полезно проводить A/A тесты? Ответьте письменно. 

*YOUR TEXT HERE*

### Exploration (0.5p)

Допишите функцию для оценки предсказаний моделей. Минимально она должна выводить график Popularity Bias (см. семинар) и топ-10 самых частотных рекомендаций с их признаками. Также интересно может быть посмотреть на преобладание каких-либо категорий/брендов и распределение стоимости айтемов в разных квантилях по частотности встречаемости в рекомендациях.

In [19]:
def explore_recommendations(model_recommendations: list[list]):
    """
    Args: 
       model_recommendations: list[list of recommended item_ids] - список рекомендаций для каждого пользователя
    """
    global ITEMS
    
    # YOUR CODE HERE

## Baselines (1.5p)

In [20]:
class BaseRecommender(ABC):
    """
    Base recommender class. All models should inherit from this.
    Defines common interface and stores column name configuration.
    """
    def __init__(self, name: str = ""):
        self.name = name
        self.user_col = "user_id"
        self.item_col = "item_id"
        self.interactions_col = "label"
        self.fitted = False

    @abstractmethod
    def _fit(self, interactions: pl.DataFrame):
        """Fit model on interaction data."""
        pass

    @abstractmethod
    def _recommend(self, user_ids: list, topn: int = 10) -> list[list]:
        """Recommend for list of users. Returns list of [item_id], len([item_id])=topn, corresponding to each user."""
        pass

    def fit(self, interactions: pl.DataFrame):
        self._fit(interactions)
        self.fitted = True

    def recommend(self, user_ids: list, topn: int = 10) -> list[list]:
        if not self.fitted:
            raise ValueError("Model is not fitted yet, call fit() first")
            
        return self._recommend(user_ids, topn)

### Random (0.5p)

Самая простая модель уже написана за вас. Обратите внимание, что разные способы сэмплирования дают разные временные оценки. Векторные операции зачастую сильно ускоряют вычисления. 

In [21]:
class RandomRecommender(BaseRecommender):
    def __init__(self):
        super().__init__(name="random")
        self.items = []

    def _fit(self, interactions: pl.DataFrame):
        self.items = interactions[self.item_col].unique().to_list()

    def _recommend(self, user_ids: list, topn: int = 10) -> list[list]:
        # a cycle + df.sample will work for 2 hours
        return np.random.choice(self.items, size=(len(user_ids), topn), replace=True)

In [22]:
model = RandomRecommender()

In [ ]:
%%time
model.fit(train_val_events)

In [ ]:
%%time
model_recommendations = model.recommend(test_users, topn=N)

In [ ]:
test_events = test_events.with_columns(pl.Series(model_recommendations).alias(model.name))
RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(evaluate(test_events, recommendations_column=model.name), index=[model.name]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

In [ ]:
explore_recommendations(model_recommendations)

В следующей ячейке письменно ответьте на вопрос, когда может пригодиться "подмешивание случайных рекомендаций" в выдачу? Как бы вы реализовали подобное подмешивание (идейно и с точки зрения способа сэмплирования)? Какие ограничения есть у предложенного подхода?

*YOUR TEXT HERE*

### TopPopular

Еще одна уже написанная моделька :)

Популярность айтема считается как суммарное количество положительных взаимодействий с ним. 

In [27]:
class TopPopularRecommender(BaseRecommender):
    def __init__(self):
        super().__init__(name="toppop")
        self.popular_items = []
        self.inner_cutoff = 200  # чтобы не хранить больше, чем потеницально нужно

    def _fit(self, interactions: pl.DataFrame):
        self.popular_items = (
            interactions
            .group_by(self.item_col).agg(pl.col(self.interactions_col).sum())
            .sort(self.interactions_col, descending=True).head(self.inner_cutoff)
            [self.item_col].to_list()
        )

    def _recommend(self, user_ids: list, topn: int = 10) -> list[list]:
        return [self.popular_items[:topn]] * len(user_ids)

In [28]:
model = TopPopularRecommender()

In [ ]:
%%time
model.fit(train_val_events)

In [ ]:
%%time
model_recommendations = model.recommend(test_users, topn=N)

In [ ]:
test_events = test_events.with_columns(pl.Series(model_recommendations).alias(model.name))
RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(evaluate(test_events, recommendations_column=model.name), index=[model.name]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

In [ ]:
explore_recommendations(model_recommendations)

### SocdemTopPopular (1p)

А вот следующую модель вам придется реализовать самостоятельно. Идея в том, чтобы посчитать свой топ популярных айтемов для каждого соцдем-кластера пользователей (используйте `group_by`). Обратите внимание, что в случае, если на инференсе кластер не определен, нужно рекомендовать глобальный топ популярных айтемов, рассчитанный без учета кластеров.

Популярность айтема считается как суммарное количество положительных взаимодействий с ним.

In [33]:
class SocdemTopPopularRecommender(BaseRecommender):
    def __init__(self):
        super().__init__(name="socdem_toppop")
        self.socdem_col = "socdem_cluster"
        self.socdem_popular_items_df = None
        self.popular_items = []
        self.inner_cutoff = 200  # чтобы не хранить больше, чем потеницально нужно

    def _fit(self, interactions: pl.DataFrame):
        # YOUR CODE HERE

    def _recommend(self, user_ids: list, topn: int = 10) -> list[list]:
        global USERS 

        # YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

## MF (3.5p)

In [39]:
class BaseMFRecommender(BaseRecommender):
    """
    Base class for matrix factorization recommenders.
    Adds user/item ID mapping and interaction matrix construction.
    """
    def __init__(self, name: str = ""):
        super().__init__(name=name)
        
        self.id2user = {}
        self.id2item = {}

        self.user2id = {}
        self.item2id = {}
        
        self.interactions_matrix: csr_matrix = None

    def _create_interaction_matrix(self, interactions: pl.DataFrame):
        """Create CSR interaction matrix and mappings. To be used in _fit() method"""
        unique_users = interactions[self.user_col].unique().to_list()
        unique_items = interactions[self.item_col].unique().to_list()
        
        self.id2user = dict(enumerate(unique_users))
        self.id2item = dict(enumerate(unique_items))
        self.user2id = {v: k for k, v in self.id2user.items()}
        self.item2id = {v: k for k, v in self.id2item.items()}
        
        rows = [self.user2id[user] for user in interactions[self.user_col]]
        cols = [self.item2id[item] for item in interactions[self.item_col]]
        
        self.interactions_matrix = csr_matrix(
            (interactions[self.interactions_col], (rows, cols)),
            shape=(len(unique_users), len(unique_items)),
            dtype=np.float32,  # лучше всегда указывать тип, чтобы не было проблем с памятью
        )

### Matrix operations (0.5p)

Простая задачка на вспомнить, как работать с матрицами. Реализуйте TopPopularRecommender как наследника класса BaseMFRecommender. Для расчетов популярных айтемов используйте матрицу интеракций вместо исходной polars-таблицы. Проверьте, что результаты совпадают с обычным TopPopularRecommender.

In [45]:
# YOUR CODE HERE

Здесь могло бы быть задание на реализацию SVD в качестве рекомедательной модели, но на практике она скорее не используется, поэтому - для желающих - остается на самостоятельное изучение :)

### iALS (2p)

#### Model (1p)

А вот что используется на практике, так это модель iALS из библиотеки `implicit`. Реализуйте iALSRecommender(BaseMFRecommender) как обертку для данной модели. Обратите внимание, что модель из коробки умеет рекомендовать для батча пользователей.

In [46]:
# YOUR CODE HERE

Подберите лучшие гиперпарметры `factors`, `regularization`, `alpha` на валидационной выборке. Замерьте не только метрики качества, но и время обучения и инференса моделей.

Обучите лучшую модель на train+val и сделайте предсказания для test так же, как с предыдущими моделями. 

In [ ]:
# YOUR CODE HERE

#### Confidence (1p)

Поэкспериментируйте с confidence-матрицей. Например, можно взвесить `label` в зависимости от `timestamp` или сделать `label` небинарным на основе `action_type`. Также можно взвесить `label` в зависимости от поверхности `subdomain`. 

Обучите iALS. Сделайте выводы.

In [ ]:
# YOUR CODE HERE

*YOUR TEXT HERE*

### iALS+ANN (1p)

#### FaissModel (0.5p)

Напишите iALSANNRecommender, который будет использовать FaissModel (тоже из `implicit`) поверх AlternatingLeastSquares для ускорения процесса рекомендаций. 

Оцените метрики качества и время работы в сравнении с AlternatingLeastSquares. Опишите результаты. Когда использование ANN может быть оправдано?

In [60]:
# YOUR CODE HERE

*YOUR TEXT HERE*

#### Item2item (0.5p)

Отвлечемся от user2item рекомендации и посмотрим в сторону item2item ("похожие товары"). Сначала взглянем, насколько похожие айтемы выдает ANN с предыдущего шага.

In [66]:
# код можно адаптировать, если у вас иная реализация
# и айдишник айтема можно менять - посмотрите-потыкайтесь
HTML(
    ITEMS.filter(
        pl.col("item_id").is_in(
            [model.id2item[idx] for idx in model.ann_model.similar_items(model.item2id["nfmcg_6755174"])[0]]
        )
    )
    .select("item_id", "category", "subcategory")
    .to_pandas().to_html()
)

,item_id,category,subcategory
0,nfmcg_11101818,"Fashion Accessories, Tech Add-ons, and Style Enhancements",Jewelry and Costume Jewelry
1,nfmcg_11635337,Electronic Devices and Gadgets,Mobile Devices and Electronic Accessories
2,nfmcg_13962039,Electronic Devices and Gadgets,Mobile Devices and Electronic Accessories
3,nfmcg_1595075,Electronic Devices and Gadgets,Mobile Devices and Electronic Accessories
4,nfmcg_19023598,Electronic Devices and Gadgets,Portable Electronics
5,nfmcg_22604333,"Cosmetics, Personal Care, and Health Maintenance Products",Perfumes and Aromatic Products
6,nfmcg_2858251,Electronic Devices and Gadgets,Mobile Devices and Electronic Accessories
7,nfmcg_39238,Home/Office Furniture and Interior Decor,Cabinets and Storage Systems
8,nfmcg_5209579,Electronic Devices and Gadgets,Mobile Devices and Electronic Accessories
9,nfmcg_6755174,Electronic Devices and Gadgets,Mobile Devices and Electronic Accessories


Теперь поработаем с теми айтемами из каталога, у которых присутствует категория (можно и без категории, но вряд ли по null будет сильно понятно, что это за айтем).

In [67]:
items_with_categories = (
    ITEMS.filter(pl.col("subcategory").is_not_null()).select("item_id", "category", "subcategory", "embedding").with_row_index()
)
items_with_categories

index,item_id,category,subcategory,embedding
u32,str,str,str,"array[f32, 300]"
0,"""nfmcg_1000002""","""Fashion Accessories, Tech Add-…","""Hats, Scarves, and Shawls""","[-0.056533, 0.082878, … 0.037475]"
1,"""nfmcg_10000039""","""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""","[-0.157201, 0.026234, … 0.013904]"
2,"""nfmcg_10000085""","""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""","[-0.124708, 0.037677, … -0.017627]"
3,"""nfmcg_10000098""","""Fashion Accessories, Tech Add-…","""Bags, Backpacks, and Travel Ac…","[-0.085646, 0.032929, … 0.069084]"
4,"""nfmcg_10000100""","""Cosmetics, Personal Care, and …","""Dental and Gum Hygiene Product…","[-0.05772, 0.081778, … 0.034646]"
…,…,…,…,…
1092381,"""nfmcg_9999917""","""Pharmaceuticals and Medical Su…","""Dietary Supplements and Vitami…","[-0.041579, 0.028325, … 0.068679]"
1092382,"""nfmcg_999993""","""Outerwear, Casual Apparel, and…","""Men's Clothing""","[-0.038017, 0.05173, … 0.097221]"
1092383,"""nfmcg_9999949""","""Outerwear, Casual Apparel, and…","""Women's Clothing""","[-0.097365, 0.038201, … 0.030096]"


Обучите FAISS-индекс на эмбеддингах айтемов. Придется немного покопаться в документации https://github.com/facebookresearch/faiss, но решение должно занять около 5 строчек кода. 

В примере ниже использовался тип индекса "IVF256_HNSW32,Flat". 

In [68]:
# YOUR CODE HERE

In [69]:
%%time
distances, indices = index.search(items_with_categories.filter(pl.col("item_id").eq("nfmcg_6755174"))["embedding"].to_numpy(), 10)
items_with_categories.filter(pl.col("index").is_in(indices.flatten()))

CPU times: user 27 ms, sys: 7.79 ms, total: 34.8 ms
Wall time: 14.8 ms


index,item_id,category,subcategory,embedding
u32,str,str,str,"array[f32, 300]"
33807,"""nfmcg_1078940""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.091484, 0.049256, … 0.041291]"
108090,"""nfmcg_12551464""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.083419, 0.047695, … 0.048635]"
288831,"""nfmcg_16815234""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.086212, 0.031782, … 0.065634]"
321776,"""nfmcg_17600235""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.070968, 0.038067, … 0.052249]"
558311,"""nfmcg_23178267""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.085651, 0.035771, … 0.040709]"
788302,"""nfmcg_2858251""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.089298, 0.038707, … 0.043044]"
954950,"""nfmcg_6755174""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.07863, 0.045097, … 0.052823]"
1013775,"""nfmcg_8139701""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.091742, 0.052721, … 0.043532]"
1021323,"""nfmcg_8317031""","""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …","[-0.083585, 0.034589, … 0.058617]"


Кстати, вместо ANN можно попробовать использовать честный kNN, но, учитывая размеры каталога, это будет долго и вероятно убьет RAM.

## NeuMF (1.5p)

На основе материалов семинара реализуйте NeuMF. Оцените качество и сравните с остальными моделями.

Часть с обучением продублирована из кода семинара, вы можете изменять её, а также подобрать гиперпараметры. Попробуйте добиться сопоставимого с предыдущими моделями качества. Опишите сделанные изменения и результаты экспериментов.

In [70]:
unique_users = train_val_events["user_id"].unique()
unique_items = train_val_events["item_id"].unique()

id2user = dict(enumerate(unique_users))
id2item = dict(enumerate(unique_items))

user2id = {v: k for k, v in id2user.items()}
item2id = {v: k for k, v in id2item.items()}

In [71]:
print(len(unique_users), len(unique_items))

9612 27205


In [72]:
train_df = (
    train_val_events
    .filter(pl.col("label") > 0)
    .with_columns(
        user_id_encoded=pl.col("user_id").replace(user2id).cast(pl.Int32), 
        item_id_encoded=pl.col("item_id").replace(item2id).cast(pl.Int32)
    )
)

user_interactions = dict(
    train_df.group_by("user_id_encoded").agg(pl.col("item_id_encoded")).iter_rows()
)  # encoded_user_id: list[encoded_item_id]

In [73]:
train_encoded_users = torch.Tensor(train_df["user_id_encoded"]).long()
train_encoded_items = torch.Tensor(train_df["item_id_encoded"]).long()

In [74]:
class NeuMF(nn.Module):
    def __init__(
        self,
        n_users: int,
        m_items: int,
        n_factors: int,
        hidden_dim: int,
    ) -> None:
        super().__init__()
        self.name = "NeuMF"

        self.m_items = m_items

        # Embeddings for GMF
        self.user_emb_gmf = nn.Embedding(n_users, n_factors)
        self.item_emb_gmf = nn.Embedding(m_items, n_factors)

        # Embeddings for MLP
        self.user_emb_mlp = nn.Embedding(n_users, n_factors)
        self.item_emb_mlp = nn.Embedding(m_items, n_factors)

        # MLP tower
        self.mlp = nn.Sequential(
            nn.Linear(2 * n_factors, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )

        # Final prediction layer
        self.out = nn.Linear(
            n_factors + hidden_dim // 2, 1
        )

        self._init_weights()

    def _init_weights(self):
        for emb in [
            self.user_emb_gmf,
            self.item_emb_gmf,
            self.user_emb_mlp,
            self.item_emb_mlp,
        ]:
            nn.init.normal_(emb.weight, std=0.01)

        for m in self.mlp:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

        nn.init.xavier_uniform_(self.out.weight)

    def forward(self, item: torch.Tensor, user: torch.Tensor) -> torch.Tensor:
        """
        item, user: LongTensor of shape (batch,)
        """

        # --- GMF part ---
        u_gmf = self.user_emb_gmf(user)      # (B, k)
        i_gmf = self.item_emb_gmf(item)      # (B, k)
        gmf = u_gmf * i_gmf                  # element-wise product

        # --- MLP part ---
        u_mlp = self.user_emb_mlp(user)      # (B, k)
        i_mlp = self.item_emb_mlp(item)      # (B, k)
        mlp_input = torch.cat([u_mlp, i_mlp], dim=-1)
        mlp_out = self.mlp(mlp_input)        # (B, hidden//2)

        # --- Fusion ---
        x = torch.cat([gmf, mlp_out], dim=-1)
        prediction = self.out(x).squeeze(-1)  # (B,)

        return prediction

In [75]:
model = NeuMF(n_users=len(unique_users), m_items=len(unique_items), n_factors=20, hidden_dim=64)
model

NeuMF(
  (user_emb_gmf): Embedding(9612, 20)
  (item_emb_gmf): Embedding(27205, 20)
  (user_emb_mlp): Embedding(9612, 20)
  (item_emb_mlp): Embedding(27205, 20)
  (mlp): Sequential(
    (0): Linear(in_features=40, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (out): Linear(in_features=52, out_features=1, bias=True)
)

In [76]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

all_items = set(range(model.m_items))

In [ ]:
num_epochs = 3
window_size = 100

for epoch in range(num_epochs):
    loss_window = deque(maxlen=window_size)
    
    pbar = tqdm(
        zip(train_encoded_users, train_encoded_items),
        total=len(train_encoded_users),
        desc=f"Epoch {epoch + 1}"
    )
    
    for user, item in pbar:
        optimizer.zero_grad()

        # negative sampling
        seen_items = set(user_interactions[user.item()])
        unseen_items = list(all_items - seen_items)
        neg_item = torch.tensor(
            np.random.choice(unseen_items),
            dtype=torch.long
        )

        # predictions
        pos_prediction = model(item.unsqueeze(0), user.unsqueeze(0))
        neg_prediction = model(neg_item.unsqueeze(0), user.unsqueeze(0))

        # targets
        pos_target = torch.ones_like(pos_prediction)
        neg_target = torch.zeros_like(neg_prediction)

        cur_loss = (
            loss_fn(pos_prediction, pos_target) 
            + loss_fn(neg_prediction, neg_target)
        )

        cur_loss.backward()
        optimizer.step()

        loss_window.append(cur_loss.item())
        avg_loss = np.mean(loss_window)

        pbar.set_postfix(loss=f"{avg_loss:.4f}")

In [ ]:
# YOUR CODE HERE

*YOUR TEXT HERE*

## Results (0.5p)

Опишите результаты, полученные в данной работе. Достаточно показать итоговую таблицу с метриками и проинтерпретировать полученные значения. Жалобы/предложения также принимаются :)

Пример того, что может получиться. Это ок, если метрики бейзлайна окажутся лучше модельных метрик, но проинтерпретировать данный феномен всё равно необходимо.

In [85]:
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,hit_rate@5,recall@5,ndcg@5,coverage@5,hit_rate@10,recall@10,ndcg@10,coverage@10,hit_rate@20,recall@20,ndcg@20,coverage@20
A/A,1.00000,0.82196,1.00000,0.00209,1.00000,0.96464,1.00000,0.00274,1.00000,0.99753,1.00000,0.00299
random,0.00172,0.00030,0.00054,0.00551,0.00287,0.00047,0.00053,0.00842,0.00430,0.00062,0.00056,0.01078
toppop,0.12815,0.02678,0.03313,0.00000,0.16657,0.03395,0.03228,0.00000,0.45269,0.11766,0.06623,0.00001
socdem_toppop,0.15510,0.03240,0.03483,0.00001,0.29702,0.07091,0.04976,0.00002,0.46502,0.12835,0.07196,0.00005
iALS,0.15224,0.03589,0.04016,0.00002,0.24713,0.05900,0.04729,0.00004,0.24713,0.05900,0.04729,0.00004
iALS-ANN,0.09576,0.02341,0.03043,0.00024,0.11669,0.03026,0.03028,0.00044,0.11669,0.03026,0.03028,0.00044
NeuMF,0.12815,0.02678,0.04990,0.00000,0.23481,0.05326,0.05699,0.00000,0.45212,0.11870,0.08157,0.00001


In [1]:
# YOUR CODE HERE

*YOUR TEXT HERE*